In [1]:
import newton
from newton.solvers.experimental.coupled import SolverCoupled, SolverCoupledProxy
from newton.solvers import SolverMuJoCo, SolverVBD
import numpy as np
import warp as wp
import math

newton.use_coord_layout_targets = True

In [2]:
# Helper functions

import json
import os
import time
import uuid
from html import escape
from pathlib import Path

from IPython.display import HTML, Javascript, display

# Optional: force CPU for debugging (default is GPU if available)
use_cpu = False
if use_cpu:
    wp.set_device("cpu")


def make_viewer(name: str, fixed_dpr: float = 2.0):
    """Create a replayable Viser recording for an output cell.

    ``fixed_dpr`` pins the playback iframe to a fixed devicePixelRatio (via the
    ``?fixedDpr=`` URL param the viser client reads from ``DevSettingsStore``).
    Patching the shared ``_camera_query_from_request`` staticmethod lets the
    param ride through both the live-server iframe URL and the Sphinx static
    embed.
    """
    recording_path = Path("../_static/recordings") / f"{name}.viser"
    recording_path.parent.mkdir(parents=True, exist_ok=True)
    viewer = newton.viewer.ViewerViser(verbose=False, record_to_viser=str(recording_path))

    cls = newton.viewer.ViewerViser
    if not hasattr(cls, "_orig_camera_query_from_request"):
        cls._orig_camera_query_from_request = cls._camera_query_from_request

    def _patched(camera_request):
        return cls._orig_camera_query_from_request(camera_request) + f"&fixedDpr={fixed_dpr}"

    cls._camera_query_from_request = staticmethod(_patched)

    return viewer


def render_mermaid(diagram: str, theme: str = "forest", line_color: str = "#76b900", width: str = "100%"):
    """Render Mermaid text as a diagram in Jupyter output."""
    element_id = f"mermaid-{uuid.uuid4().hex}"
    diagram_html = escape(diagram)
    width_style = escape(width, quote=True)
    display(HTML(f'<pre id="{element_id}" class="mermaid" style="width: {width_style};">{diagram_html}</pre>'))

    if os.environ.get("NEWTON_SPHINX_BUILD") == "1":
        # sphinxcontrib-mermaid renders pre.mermaid blocks in the built docs.
        return

    config_json = json.dumps(
        {
            "startOnLoad": False,
            "theme": theme,
            "themeVariables": {"lineColor": line_color},
        }
    )

    js = f"""
(async () => {{
  const loadMermaid = () => new Promise((resolve, reject) => {{
    if (window.mermaid) {{
      resolve();
      return;
    }}
    const script = document.createElement("script");
    script.src = "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js";
    script.onload = () => resolve();
    script.onerror = () => reject(new Error("Failed to load Mermaid."));
    document.head.appendChild(script);
  }});

  await loadMermaid();
  const config = {config_json};
  window.mermaid.initialize(config);

  const node = document.getElementById("{element_id}");
  if (!node) {{
    return;
  }}

  try {{
    await window.mermaid.run({{ nodes: [node] }});
  }} catch (err) {{
    node.textContent = "Mermaid render error: " + (err && err.message ? err.message : err);
  }}
}})();
"""
    display(Javascript(js))


class _HTMLProgressBar:
    """Frontend-agnostic HTML progress bar for notebook loops."""

    def __init__(self, iterable, total: int, desc: str = "", leave: bool = True):
        self._iterable = iterable
        self.total = max(int(total), 1)
        self.desc = desc
        self.leave = leave
        self.n = 0
        self._start = time.perf_counter()
        self._last_refresh = 0.0
        self._handle = display(HTML(self._render()), display_id=True)

    def _render(self) -> str:
        elapsed = max(time.perf_counter() - self._start, 1e-9)
        pct = min(100.0, 100.0 * self.n / self.total)
        rate = self.n / elapsed
        remaining = (self.total - self.n) / rate if rate > 1e-9 else float("inf")
        eta = "--:--" if not np.isfinite(remaining) else f"{int(remaining // 60):02d}:{int(remaining % 60):02d}"
        desc_html = escape(self.desc)
        return (
            f'<div style="font-family: sans-serif; margin: 6px 0;">'
            f'<div style="display:flex; justify-content:space-between; font-size:12px; margin-bottom:4px;">'
            f"<span>{desc_html}</span>"
            f"<span>{self.n}/{self.total} ({pct:5.1f}%)</span>"
            f"</div>"
            f'<progress value="{self.n}" max="{self.total}" style="width:100%; height:14px;"></progress>'
            f'<div style="font-size:11px; color:#666; margin-top:2px;">elapsed {elapsed:5.1f}s | eta {eta}</div>'
            f"</div>"
        )

    def _refresh(self, force: bool = False):
        now = time.perf_counter()
        if force or (now - self._last_refresh) >= 0.1 or self.n >= self.total:
            self._handle.update(HTML(self._render()))
            self._last_refresh = now

    def set_description(self, desc: str):
        self.desc = desc
        self._refresh(force=True)

    def set_description_str(self, desc: str):
        self.set_description(desc)

    def update(self, n: int = 1):
        self.n = min(self.total, self.n + int(n))
        self._refresh()

    def close(self):
        if self.leave:
            self._refresh(force=True)
        else:
            self._handle.update(HTML(""))

    def __iter__(self):
        try:
            for item in self._iterable:
                yield item
                self.update(1)
        finally:
            self.close()


def _tqdm_html(iterable=None, total=None, desc: str = "", leave: bool = False, **_kwargs):
    if iterable is None:
        if total is None:
            raise ValueError("Either iterable or total must be provided.")
        iterable = range(int(total))
    if total is None:
        try:
            total = len(iterable)
        except TypeError as e:
            raise ValueError("total is required for iterables without len().") from e
    return _HTMLProgressBar(iterable=iterable, total=int(total), desc=desc, leave=leave)


def _trange_html(*args, **kwargs):
    return _tqdm_html(range(*args), **kwargs)


# Use HTML progress bars for reliable updates in this frontend.
tqdm, trange = _tqdm_html, _trange_html


AXIS_COLORS = wp.array([(1.0, 0.1, 0.1), (0.1, 1.0, 0.1), (0.1, 0.4, 1.0)], dtype=wp.vec3)
AXIS_HALF = 0.05
_AXIS_BASIS = (wp.vec3(1.0, 0.0, 0.0), wp.vec3(0.0, 1.0, 0.0), wp.vec3(0.0, 0.0, 1.0))


def log_frame_axes(viewer, name, pos, quat=None, *, half=AXIS_HALF, width=0.01):
    """Log a 3-line RGB axis indicator at ``pos`` rotated by xyzw ``quat``.

    ``pos`` may be a 3-vec or a 7-element ``wp.transform``; in the latter case
    the rotation is read from the transform and ``quat`` is ignored.
    """
    arr = np.asarray(pos, dtype=np.float32).reshape(-1)
    if arr.size == 7:
        position, rotation = arr[:3], arr[3:7]
    elif arr.size == 3:
        position = arr
        rotation = quat if quat is not None else (0.0, 0.0, 0.0, 1.0)
    else:
        raise ValueError(f"log_frame_axes: pos must be length 3 or 7, got {arr.size}")
    q = wp.quat(*(float(c) for c in rotation))
    starts = np.tile(position, (3, 1))
    ends = np.stack([position + half * np.asarray(wp.quat_rotate(q, b), dtype=np.float32) for b in _AXIS_BASIS])
    viewer.log_lines(name, starts, ends, colors=AXIS_COLORS, width=width)

Warp 1.16.0 initialized:
   CUDA not enabled in this build
   Devices:
     "cpu"      : "arm"
   Kernel cache:
     /Users/runzhang/Library/Caches/warp/1.16.0


In [3]:
FRANKA_FINGER_CLOSED = 0.0
FRANKA_FINGER_OPEN = 0.04

# Franka initial joint positions (7 arm joints [rad] + 2 gripper fingers [m])
FRANKA_HOME_Q = [
    0.0,
    0.02,
    0.0,
    -2.37,
    0.0,
    2.39,
    np.pi / 4,
    FRANKA_FINGER_OPEN,
    FRANKA_FINGER_OPEN,
]

FRANKA_DOF_COUNT = 9
FRANKA_ARM_DOF_COUNT = 7

FRANKA_TARGET_KE = [900, 900, 700, 700, 400, 400, 400, 100, 100]
FRANKA_TARGET_KD = [90, 90, 70, 70, 40, 40, 40, 10, 10]

# These values set controller limits and solver regularization for the first 9 Franka DOFs.
FRANKA_EFFORT_LIMITS = [87, 87, 87, 87, 12, 12, 12, 100, 100]
FRANKA_MUJOCO_ARMATURE = [0.195] * 4 + [0.074] * 3 + [0.1] * 2

In [32]:
# Do the cable
NUM_ELEMENTS = 60
END_SEPARATION = 0.50
TOP_HEIGHT = 0.25
SAG_DEPTH = 0.05

BEND_STIFFNESS = 0.05
BEND_DAMPING = 0.6
TWIST_STIFFNESS = 2.0
TWIST_DAMPING = 0.6

CONTACT_TOPOLOGICAL_FILTER_SPAN = 2

def _hanging_arc_nodes() -> np.ndarray:
    u = np.linspace(0.0, 1.0, NUM_ELEMENTS + 1)
    x = (u - 0.5) * END_SEPARATION
    y = np.zeros_like(u)
    z = TOP_HEIGHT - SAG_DEPTH * np.sin(math.pi * u)
    return np.column_stack([x, y, z]).astype(np.float64)

def _filter_near_rod_collision_pairs(builder, bodies: list[int], span: int) -> None:
    for i, body_i in enumerate(bodies):
        for j in range(i + 1, min(len(bodies), i + span + 1)):
            body_j = bodies[j]
            for shape_i in builder.body_shapes.get(body_i, []):
                for shape_j in builder.body_shapes.get(body_j, []):
                    builder.add_shape_collision_filter_pair(int(shape_i), int(shape_j))

nodes = _hanging_arc_nodes()
seg_lengths = np.linalg.norm(np.diff(nodes, axis=0), axis=1)
segment_length = float(np.mean(seg_lengths))
cable_radius = 0.003
contact_gap = 0.001

points = [wp.vec3(*p) for p in nodes]

builder = newton.ModelBuilder(gravity=(0.0, 0.0, -9.81))

robot_base_pos = wp.vec3(-0.5, 0.0, 0.0)
builder.add_urdf(
    str("../assets/franka_emika_panda/urdf/fr3_franka_hand.urdf"),
    xform=wp.transform(robot_base_pos, wp.quat_identity()),
    floating=False,
    enable_self_collisions=False,
    parse_visuals_as_colliders=False,
)

builder.joint_q[:FRANKA_DOF_COUNT] = FRANKA_HOME_Q
builder.joint_effort_limit[:FRANKA_DOF_COUNT] = FRANKA_EFFORT_LIMITS
builder.joint_armature[:FRANKA_DOF_COUNT] = FRANKA_MUJOCO_ARMATURE
builder.joint_target_ke[:FRANKA_DOF_COUNT] = FRANKA_TARGET_KE
builder.joint_target_kd[:FRANKA_DOF_COUNT] = FRANKA_TARGET_KD


newton.solvers.SolverMuJoCo.register_custom_attributes(builder)
builder.validate_inertia_detailed = True

table_pos = wp.vec3(0.0, 0.0, 0.5 * table_height)
table_body = builder.add_body(xform=wp.transform(table_pos, wp.quat_identity()))
builder.add_joint_fixed(parent=-1, child=table_body)

# Attach table box to table_body instead of body=-1
builder.add_shape_box(
    body=table_body,
    hx=0.4, hy=0.4, hz=0.5 * table_height,
    xform=wp.transform_identity(),
    cfg=newton.ModelBuilder.ShapeConfig(mu=0.8)
)

cube_size = 0.05
cube_body = None
drop_offset = 0.05
cube_pos = wp.vec3(0.0, 0.15, table_height + 0.5 * cube_size + drop_offset)
cube_body = builder.add_body(xform=wp.transform(cube_pos, wp.quat_identity()))
shape_cfg = newton.ModelBuilder.ShapeConfig(margin=0.0, density=400.0)
builder.add_shape_box(
    body=cube_body,
    hx=0.5 * cube_size,
    hy=0.5 * cube_size,
    hz=0.5 * cube_size,
    cfg=shape_cfg,
)

shape_cfg = newton.ModelBuilder.ShapeConfig(
    ke = 1.0e4,
    kd = 50,
    gap = contact_gap,
    mu = 0.8,
    density=1000.0
)

rigid_body_ids = [b for b in range(builder.body_count)]   # Robot, Cube, Table
rigid_shape_ids = list(range(builder.shape_count))        # Shapes for Robot, Cube, Table

bodies, joints = builder.add_rod(
    positions = points,
    quaternions = None,
    radius = cable_radius,
    cfg = shape_cfg,
    stretch_stiffness=1.0e6,
    stretch_damping=6.0,
    bend_stiffness=BEND_STIFFNESS,
    bend_damping=BEND_DAMPING,
    twist_stiffness=TWIST_STIFFNESS,
    twist_damping=TWIST_DAMPING,
    label="cable",
    wrap_in_articulation=False,
    body_frame_origin="com",
    color = (0.05, 0.15, 0.65, 1.0),
)


soft_body_ids = list(map(int, bodies))
cable_shape_ids = [s for s in range(builder.shape_count) if s not in rigid_shape_ids]


joints = list(map(int, joints))
_filter_near_rod_collision_pairs(builder, bodies, CONTACT_TOPOLOGICAL_FILTER_SPAN)


builder.add_articulation(joints, label="cable_articulation")

builder.color()

/var/folders/p3/t1jhxtjn3h704zlkrgnk5mlr0000gn/T/ipykernel_43367/33496291.py:60: UserWarning: Adding a FIXED joint between parent -1 and child 14 (label: 'body_14'), but another joint already connects these bodies. A FREE joint parallel to another joint is inconsistent. Use add_link() with the appropriate joint type instead of add_body().
  builder.add_joint_fixed(parent=-1, child=table_body)


In [20]:
soft_shape_ids = cable_shape_ids + table_shape_ids

In [23]:
print("Total joints:", builder.joint_count)
print(rigid_body_ids)
print(soft_body_ids)
print(rigid_shape_ids)

# Table is attached to static body -1
table_shape_ids = builder.body_shapes.get(-1, [])   # Contains [54]
cable_shape_ids = [s for s in range(builder.shape_count) if s not in rigid_shape_ids]

print(table_shape_ids, cable_shape_ids)

Total joints: 74
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55]
[54] [54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]


In [24]:
print(rigid_shape_ids)
print(soft_shape_ids)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55]
[54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 54]


In [25]:
print(rigid_body_ids)
print(soft_body_ids)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74]


In [27]:
total_shapes = list(range(builder.shape_count))

# 2. Extract table shape (body -1)
table_shape_ids = list(builder.body_shapes.get(-1, []))  # [54]

# 3. Dynamic rigid shapes (Robot + Cube, excluding static table)
# Shapes 0..53 (robot) and 55 (cube)
rigid_shape_ids = [s for s in range(56) if s not in table_shape_ids]

# 4. Pure cable shapes (56..115)
cable_shape_ids = [s for s in total_shapes if s >= 56]

# 5. Soft entry shapes: Cable + Table (without duplicates)
soft_shape_ids = sorted(list(set(cable_shape_ids + table_shape_ids)))

print("rigid_shape_ids:", rigid_shape_ids)  # [0..53, 55]
print("soft_shape_ids:", soft_shape_ids)    # [54, 56..115]

rigid_shape_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55]
soft_shape_ids: [54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]


In [33]:
arm_joint_ids = list(range(15))
model = builder.finalize()

collision_pipeline = newton.CollisionPipeline(model, contact_matching="latest")
contacts = collision_pipeline.contacts()
sim_iterations=20

rigid_entry = SolverCoupled.Entry(
    name = "rigid",
    solver = lambda view: SolverMuJoCo(model=view,njmax=256, nconmax=256,),
    bodies=rigid_body_ids,
    joints=arm_joint_ids,
    shapes=rigid_shape_ids,
    substeps=1,
)

cable_shape_ids = [s for s in range(builder.shape_count) if s not in rigid_shape_ids]

soft_entry = SolverCoupled.Entry(
    name="soft",
    solver=lambda view: SolverVBD(
        model=view,
        iterations=20,
        rigid_contact_hard=True,
        rigid_contact_history=True,
        rigid_body_contact_buffer_size=1024,
    ),
    bodies=soft_body_ids,
    shapes=cable_shape_ids,
    joints=joints,
    substeps=2,
)

solver = SolverCoupledProxy(
    model,
    entries=[rigid_entry, soft_entry],
    coupling=SolverCoupledProxy.Config(
        proxies=[
            SolverCoupledProxy.Proxy(
                source="rigid",
                destination="soft",
                bodies=rigid_body_ids,
                proxy_bodies=None,  # Reuses source IDs in destination view
                mass_scale=1.0,
                mode="staggered",   # Staggered mode uses latest arm/cube positions
                proxy_relaxation=0.8,
            )
        ],
        iterations=2,
    ),
)


state_0 = model.state()
state_1 = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state_0)
state_1.body_q.assign(state_0.body_q)
control = model.control()
joint_targets = control.joint_target_q.numpy()
joint_targets[:FRANKA_DOF_COUNT] = FRANKA_HOME_Q
control.joint_target_q.assign(joint_targets)

ValueError: SolverMuJoCo cannot convert bodies that are outside articulations and have no standalone joint to world. Bodies: ['body_15']. Related joints: [].

In [29]:

fps = 60
duration = 3.0  # seconds
num_frames = int(fps * duration)
frame_dt = 1.0 / fps
sim_substeps = 8
sim_dt = frame_dt / sim_substeps

viewer = make_viewer("cable_drop_test")
viewer.set_model(model)
viewer.set_camera(wp.vec3(0.0, -1.0, 0.5), -15, 90)

sim_time = 0.0

for frame in range(num_frames):
    # Log state for visualization
    viewer.begin_frame(sim_time)
    viewer.log_state(state_0)
    viewer.log_contacts(contacts, state_0)
    viewer.end_frame()

    # Step simulation across substeps (CPU execution)
    for _ in range(sim_substeps):
        state_0.clear_forces()
        viewer.apply_forces(state_0)

        collision_pipeline.collide(state_0, contacts)

        # solver.set_rigid_history_update(True)
        solver.step(state_0, state_1, control, contacts, sim_dt)

        state_0, state_1 = state_1, state_0

    sim_time += frame_dt

viewer


╭────── viser (listening *:8082) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8082   │
│   Websocket │ ws://localhost:8082     │
│             ╵                         │
╰───────────────────────────────────────╯